# Fine-tune DistilBERT for Question Answering

This notebook fine-tunes a Transformer QA model on a small SQuAD v1.1 subset and saves it for the app. Internet is needed to download the model and dataset. CPU training may take a while.

In [1]:
%pip install -q transformers datasets accelerate torch

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
%pip install --upgrade datasets huggingface_hub

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os, re
import numpy as np
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, TrainingArguments, Trainer, default_data_collator

c:\Users\iqrak\SK\DL and  Gen AI\StudyDoc-AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
TRAIN_SAMPLES = 2000  # Increase only if your machine can handle it
VAL_SAMPLES = 250
MODEL_NAME = "distilbert/distilbert-base-uncased"
OUTPUT_DIR = "../models/studydoc-qa"
MAX_LENGTH = 384
STRIDE = 96

from datasets import load_dataset

raw = load_dataset("rajpurkar/squad")

train_data = raw["train"].shuffle(seed=42).select(
    range(min(TRAIN_SAMPLES, len(raw["train"])))
)

val_data = raw["validation"].shuffle(seed=42).select(
    range(min(VAL_SAMPLES, len(raw["validation"])))
)

print("Train examples:", len(train_data))
print("Validation examples:", len(val_data))

Train examples: 2000
Validation examples: 250


In [10]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 2447.83it/s]
[transformers] DistilBertForQuestionAnswering LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
qa_outputs.bias         | MISSING    | 
qa_outputs.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
def prepare_features(examples):
    questions = [q.strip() for q in examples["question"]]
    enc = tokenizer(questions, examples["context"], max_length=MAX_LENGTH, truncation="only_second", stride=STRIDE, return_overflowing_tokens=True, return_offsets_mapping=True, padding="max_length")
    sample_map = enc.pop("overflow_to_sample_mapping")
    offsets_all = enc.pop("offset_mapping")
    starts, ends = [], []
    for i, offsets in enumerate(offsets_all):
        input_ids = enc["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)
        sequence_ids = enc.sequence_ids(i)
        sample_index = sample_map[i]
        answers = examples["answers"][sample_index]
        if not answers["answer_start"]:
            starts.append(cls_index); ends.append(cls_index); continue
        start_char = answers["answer_start"][0]
        end_char = start_char + len(answers["text"][0])
        context_tokens = [j for j, sid in enumerate(sequence_ids) if sid == 1]
        if not context_tokens:
            starts.append(cls_index); ends.append(cls_index); continue
        first, last = context_tokens[0], context_tokens[-1]
        if offsets[first][0] > start_char or offsets[last][1] < end_char:
            starts.append(cls_index); ends.append(cls_index); continue
        while first < len(offsets) and offsets[first][0] <= start_char: first += 1
        while last >= 0 and offsets[last][1] >= end_char: last -= 1
        starts.append(first - 1); ends.append(last + 1)
    enc["start_positions"] = starts
    enc["end_positions"] = ends
    return enc

train_features = train_data.map(prepare_features, batched=True, remove_columns=train_data.column_names)
val_features = val_data.map(prepare_features, batched=True, remove_columns=val_data.column_names)
print(train_features)

Map: 100%|██████████| 250/250 [00:00<00:00, 1329.04 examples/s]

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'start_positions', 'end_positions'],
    num_rows: 2015
})


In [13]:
training_args = TrainingArguments(
    output_dir="../results/checkpoints",
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=50,
    report_to="none",
    fp16=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_features,
    eval_dataset=val_features,
    processing_class=tokenizer,
    data_collator=default_data_collator,
)

In [14]:
trainer.train()

c:\Users\iqrak\SK\DL and  Gen AI\StudyDoc-AI\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,3.541282,3.430072


TrainOutput(global_step=252, training_loss=3.9690146502994357, metrics={'train_runtime': 1100.1642, 'train_samples_per_second': 1.832, 'train_steps_per_second': 0.229, 'total_flos': 197449497423360.0, 'train_loss': 3.9690146502994357, 'epoch': 1.0})

In [15]:
def normalize_answer(s):
    s = s.lower()
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())

def token_f1(pred, ref):
    p, r = normalize_answer(pred).split(), normalize_answer(ref).split()
    if not p or not r: return float(p == r)
    from collections import Counter
    common = Counter(p) & Counter(r)
    overlap = sum(common.values())
    if overlap == 0: return 0.0
    precision, recall = overlap/len(p), overlap/len(r)
    return 2*precision*recall/(precision+recall)

pred = trainer.predict(val_features)
start_logits, end_logits = pred.predictions
exact, f1 = [], []
for i, feat in enumerate(val_features):
    s, e = int(np.argmax(start_logits[i])), int(np.argmax(end_logits[i]))
    guess = "" if e < s or e-s > 40 else tokenizer.decode(feat["input_ids"][s:e+1], skip_special_tokens=True).strip()
    refs = val_data[i % len(val_data)]["answers"]["text"] or [""]
    exact.append(float(any(normalize_answer(guess) == normalize_answer(r) for r in refs)))
    f1.append(max(token_f1(guess, r) for r in refs))
print(f"Approx. feature-level Exact Match: {np.mean(exact):.4f}")
print(f"Approx. feature-level Token F1: {np.mean(f1):.4f}")

Approx. feature-level Exact Match: 0.0236
Approx. feature-level Token F1: 0.0436


In [16]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Saved model to:", os.path.abspath(OUTPUT_DIR))

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  5.41it/s]


Saved model to: c:\Users\iqrak\SK\DL and  Gen AI\StudyDoc-AI\models\studydoc-qa


In [17]:
sample = val_data[0]
inputs = tokenizer(sample["question"], sample["context"], return_tensors="pt", truncation="only_second", max_length=MAX_LENGTH)
model.eval()
with torch.no_grad(): out = model(**inputs)
s, e = int(out.start_logits.argmax(-1)), int(out.end_logits.argmax(-1))
print("Question:", sample["question"])
print("Prediction:", tokenizer.decode(inputs["input_ids"][0][s:e+1], skip_special_tokens=True))
print("Reference:", sample["answers"]["text"])

Question: In what year did Massachusetts first require children to be educated in schools?
Prediction: private schooling in the united states has been debated by educators, lawmakers and parents, since the beginnings of compulsory education in massachusetts in 1852
Reference: ['1852', '1852', '1852']
